In [ ]:
"""
Example script to run the electricity model.
It consists of three parts:
1. Generation
2. Grid
3. Storage
"""

import pandas as pd
import numpy as np
import xarray as xr
import scipy
import warnings
from pathlib import Path
import warnings
from pint.errors import UnitStrippedWarning
warnings.simplefilter("ignore", UnitStrippedWarning)

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA

from imagematerials.model import GenericStocks, MaterialIntensities, SharesInflowStocks
from imagematerials.factory import ModelFactory, Sector
from imagematerials.preprocessing import get_preprocessing_data
import prism


# TODO absolute path of file "preprocessing.py" ? current solution can differ depending on IDE used (?) 
path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_base = Path(path_base, "data", "raw")

YEAR_START = 1971   # start year of the simulation period
YEAR_END = 2100     # end year of the calculations
YEAR_OUT = 2100     # year of output generation = last year of reporting

# Combined run - all electricity subsectors

In [ ]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA


    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    
    factory = ModelFactory(
    elc_sector, complete_timeline
    ).add(GenericStocks, ["elc_gen",
                         "elc_grid_lines",
                         "elc_grid_add",
                         "elc_stor_phs"
                          ]
    ).add(SharesInflowStocks, ["elc_stor_other"],
    ).add(MaterialIntensities, ["elc_gen",
                         "elc_grid_lines",
                         "elc_grid_add",
                         "elc_stor_phs",
                         "elc_stor_other"
                          ]
    )
    model = factory.finish()

    model.simulate(simulation_timeline)
    print(f"Finished {scen_id}")

# Generation -------------------------------------------------------

In [ ]:
climate_scen = "SSP2_M_CP"
circular_scen = None

time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
# climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA


elc_sector = get_preprocessing_data("electricity", path_base, climate_policy_scenario_dir, circular_economy_scenario_dir, cache = False) 
# elc_sector is a list of preprocessing data for each electricity subsector

elc_sector_gen = next(item for item in elc_sector if item.name == "elc_gen")

factory = ModelFactory(
    elc_sector_gen, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model = factory.finish()

model.simulate(simulation_timeline)
list(model.elc_gen)


# Grid -------------------------------------------------------------

In [ ]:
climate_scen = "SSP2_M_CP"
circular_scen = None

time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
# climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA


elc_sector = get_preprocessing_data("electricity", path_base,
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dir, cache = False) 
# elc_sector is a list of preprocessing data for each electricity subsector

## Lines ----

In [ ]:
elc_sector_grid_lines = next(item for item in elc_sector if item.name == "elc_grid_lines")

factory = ModelFactory(
    elc_sector_grid_lines, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_lines = factory.finish()

model_lines.simulate(simulation_timeline)
list(model_lines.elc_grid_lines)

## Grid Additions ----

In [ ]:
elc_sector_grid_add = next(item for item in elc_sector if item.name == "elc_grid_add")

factory = ModelFactory(
    elc_sector_grid_add, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_add = factory.finish()

model_add.simulate(simulation_timeline)
list(model_add.elc_grid_add)

# Storage -------------------------------------------------------

In [ ]:
climate_scen = "SSP2_M_CP"
circular_scen = None

time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
# climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA


elc_sector = get_preprocessing_data("electricity", path_base, climate_policy_scenario_dir, circular_economy_scenario_dir, cache = False) 
# elc_sector is a list of preprocessing data for each electricity subsector

In [ ]:
# Pumped Hydropower Storage (PHS) =====

elc_sector_stor_phs = next(item for item in elc_sector if item.name == "elc_stor_phs")

factory = ModelFactory(
    elc_sector_stor_phs, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_phs = factory.finish()

model_phs.simulate(simulation_timeline)
list(model_phs.elc_stor_phs)

In [ ]:
# Other Storage =====

elc_sector_stor_other = next(item for item in elc_sector if item.name == "elc_stor_other")
# sec_electr_stor_oth = Sector("electr_stor_oth", prep_data_oth_storage, check_coordinates=False)

factory = ModelFactory(
    elc_sector_stor_other, complete_timeline
    ).add(SharesInflowStocks
    ).add(MaterialIntensities
    )
model_other = factory.finish()

model_other.simulate(simulation_timeline)
list(model_other.elc_stor_other)